# Level 2 – Groupby, Joins & Reshaping
## Exercise 1 – Groupby & Crosstab
groupby() = split → apply → combine
crosstab() = frequency table between two categorical variables

In [1]:
import pandas as pd

df = pd.read_csv("financials_clean.csv")
df['Profit Margin %'] = round((df['Profit'] / df['Sales']) * 100, 2)
print(df.shape)

(688, 26)


In [2]:
sales_by_segment = df.groupby('Segment')['Sales'].sum().sort_values(ascending=False)
print("Total Sales by Segment:")
print(sales_by_segment)

Total Sales by Segment:
Segment
Government          5.250426e+07
Small Business      4.242792e+07
Enterprise          1.961169e+07
Midmarket           2.381883e+06
Channel Partners    1.800594e+06
Name: Sales, dtype: float64


In [3]:
sales_by_segment = df.groupby('Segment')['Sales'].sum().sort_values(ascending=False)
print("Total Sales by Segment:")
print(sales_by_segment.apply(lambda x: f"{x:,.2f}"))

Total Sales by Segment:
Segment
Government          52,504,260.67
Small Business      42,427,918.50
Enterprise          19,611,694.38
Midmarket            2,381,883.08
Channel Partners     1,800,593.64
Name: Sales, dtype: str


In [4]:
summary = df.groupby('Segment').agg(
    Total_Sales=('Sales', 'sum'),
    Total_Profit=('Profit', 'sum'),
    Avg_Margin=('Profit Margin %', 'mean'),
    Count=('Sales', 'count')
).round(2)
print("\nSegment Summary:")
print(summary)


Segment Summary:
                  Total_Sales  Total_Profit  Avg_Margin  Count
Segment                                                       
Channel Partners   1800593.64    1316803.14       73.02    100
Enterprise        19611694.38    -614545.62       -3.06    100
Government        52504260.67   11388173.17       29.44    288
Midmarket          2381883.08     660103.07       27.67    100
Small Business    42427918.50    4143168.50        9.67    100


In [5]:
country_segment = df.groupby(['Country', 'Segment'])['Sales'].sum().reset_index()
country_segment = country_segment.sort_values('Sales', ascending=False)
print("\nTop 10 Country + Segment combinations:")
print(country_segment.head(10))


Top 10 Country + Segment combinations:
                     Country         Segment        Sales
7                     France      Government  12127782.72
24  United States of America  Small Business  11456559.00
12                   Germany      Government  11452895.94
2                     Canada      Government  10741236.52
17                    Mexico      Government   9791599.38
4                     Canada  Small Business   9177549.00
22  United States of America      Government   8390746.11
9                     France  Small Business   7369606.50
14                   Germany  Small Business   7327848.00
19                    Mexico  Small Business   7096356.00


In [6]:
ct_count = pd.crosstab(df['Country'], df['Segment'])
print("\nCrosstab — Record count:")
print(ct_count)


Crosstab — Record count:
Segment                   Channel Partners  Enterprise  Government  Midmarket  \
Country                                                                         
Canada                                  20          20          57         20   
France                                  20          20          58         20   
Germany                                 20          20          59         20   
Mexico                                  20          20          57         20   
United States of America                20          20          57         20   

Segment                   Small Business  
Country                                   
Canada                                20  
France                                20  
Germany                               20  
Mexico                                20  
United States of America              20  


In [7]:
df.to_csv("financials_clean.csv", index=False)
print("Saved!")

Saved!


In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("financials_clean.csv")

In [2]:
df['Profit Margin %'] = round((df['Profit'] / df['Sales']) * 100, 2)
print(df.shape)

(688, 26)


In [3]:
ct_sales = pd.crosstab(
    df['Country'], 
    df['Segment'], 
    values=df['Sales'], 
    aggfunc='sum'
).round(0)
print("\nCrosstab — Total Sales:")
print(ct_sales)


Crosstab — Total Sales:
Segment                   Channel Partners  Enterprise  Government  Midmarket  \
Country                                                                         
Canada                            491164.0   3967491.0  10741237.0   510214.0   
France                            372090.0   3890891.0  12127783.0   593802.0   
Germany                           336426.0   4086826.0  11452896.0   301345.0   
Mexico                            234379.0   3315881.0   9791599.0   511136.0   
United States of America          366534.0   4350605.0   8390746.0   465386.0   

Segment                   Small Business  
Country                                   
Canada                         9177549.0  
France                         7369606.0  
Germany                        7327848.0  
Mexico                         7096356.0  
United States of America      11456559.0  


## Exercise 2 – Merge & Concat
merge() = SQL JOIN (combine by a key column)
concat() = stack DataFrames vertically or horizontally

In [5]:
products = pd.DataFrame({
    'product_id': [1, 2, 3, 4, 5],
    'product_name': ['Paseo', 'VTT', 'Velo', 'Amarilla', 'Montana'],
    'category': ['Bikes', 'Bikes', 'Accessories', 'Clothing', 'Bikes'],
    'unit_price': [120.00, 95.00, 45.00, 30.00, 150.00]
})

sales = pd.DataFrame({
    'sale_id': [1, 2, 3, 4, 5, 6, 7, 8],
    'product_id': [1, 2, 3, 1, 4, 5, 2, 3],
    'units_sold': [50, 30, 80, 20, 60, 15, 45, 25],
    'country': ['Spain', 'France', 'Germany', 'Spain', 'Canada', 'USA', 'France', 'Germany'],
    'sale_date': ['2024-01-15', '2024-02-10', '2024-03-05', '2024-04-20',
                  '2024-05-12', '2024-06-08', '2024-07-19', '2024-08-30']
})

# Inner join
merged = pd.merge(sales, products, on='product_id', how='inner')
print("Inner join result:")
print(merged[['product_name', 'category', 'units_sold', 'country', 'unit_price']])

Inner join result:
  product_name     category  units_sold  country  unit_price
0        Paseo        Bikes          50    Spain       120.0
1          VTT        Bikes          30   France        95.0
2         Velo  Accessories          80  Germany        45.0
3        Paseo        Bikes          20    Spain       120.0
4     Amarilla     Clothing          60   Canada        30.0
5      Montana        Bikes          15      USA       150.0
6          VTT        Bikes          45   France        95.0
7         Velo  Accessories          25  Germany        45.0


In [6]:
merged['total_revenue'] = merged['units_sold'] * merged['unit_price']
print("\nWith total revenue:")
print(merged[['product_name', 'units_sold', 'unit_price', 'total_revenue']])


With total revenue:
  product_name  units_sold  unit_price  total_revenue
0        Paseo          50       120.0         6000.0
1          VTT          30        95.0         2850.0
2         Velo          80        45.0         3600.0
3        Paseo          20       120.0         2400.0
4     Amarilla          60        30.0         1800.0
5      Montana          15       150.0         2250.0
6          VTT          45        95.0         4275.0
7         Velo          25        45.0         1125.0


In [7]:
extra_products = pd.DataFrame({
    'product_id': [6],
    'product_name': ['Carretera'],
    'category': ['Bikes'],
    'unit_price': [80.00]
})
products_extended = pd.concat([products, extra_products], ignore_index=True)
left_merge = pd.merge(products_extended, sales, on='product_id', how='left')
print("\nLeft join — products with no sales show NaN:")
print(left_merge[left_merge['sale_id'].isna()])


Left join — products with no sales show NaN:
   product_id product_name category  unit_price  sale_id  units_sold country  \
8           6    Carretera    Bikes        80.0      NaN         NaN     NaN   

  sale_date  
8       NaN  


In [8]:
sales_2023 = sales.copy()
sales_2023['year'] = 2023
sales_2024 = sales.copy()
sales_2024['year'] = 2024
all_sales = pd.concat([sales_2023, sales_2024], ignore_index=True)
print(f"\nCombined sales: {len(all_sales)} rows ({len(sales_2023)} + {len(sales_2024)})")


Combined sales: 16 rows (8 + 8)


## Exercise 3 – Reshaping DataFrames
pivot() = long → wide (like an unpivot in Power Query)
melt() = wide → long (useful for visualization)

In [9]:
wide_df = pd.DataFrame({
    'Country': ['Spain', 'France', 'Germany', 'Canada', 'USA'],
    'Q1_Sales': [15000, 12000, 18000, 9000, 21000],
    'Q2_Sales': [17000, 14000, 16000, 11000, 19000],
    'Q3_Sales': [14000, 13000, 20000, 10000, 22000],
    'Q4_Sales': [20000, 16000, 25000, 13000, 28000]
})
print("Wide format:")
print(wide_df)

Wide format:
   Country  Q1_Sales  Q2_Sales  Q3_Sales  Q4_Sales
0    Spain     15000     17000     14000     20000
1   France     12000     14000     13000     16000
2  Germany     18000     16000     20000     25000
3   Canada      9000     11000     10000     13000
4      USA     21000     19000     22000     28000


In [10]:
long_df = wide_df.melt(
    id_vars=['Country'],
    value_vars=['Q1_Sales', 'Q2_Sales', 'Q3_Sales', 'Q4_Sales'],
    var_name='Quarter',
    value_name='Sales'
)
print("\nLong format after melt:")
print(long_df.sort_values(['Country', 'Quarter']).head(12))


Long format after melt:
    Country   Quarter  Sales
3    Canada  Q1_Sales   9000
8    Canada  Q2_Sales  11000
13   Canada  Q3_Sales  10000
18   Canada  Q4_Sales  13000
1    France  Q1_Sales  12000
6    France  Q2_Sales  14000
11   France  Q3_Sales  13000
16   France  Q4_Sales  16000
2   Germany  Q1_Sales  18000
7   Germany  Q2_Sales  16000
12  Germany  Q3_Sales  20000
17  Germany  Q4_Sales  25000


In [11]:
long_df['Quarter'] = long_df['Quarter'].str.replace('_Sales', '')
print("\nCleaned Quarter column:")
print(long_df.head())


Cleaned Quarter column:
   Country Quarter  Sales
0    Spain      Q1  15000
1   France      Q1  12000
2  Germany      Q1  18000
3   Canada      Q1   9000
4      USA      Q1  21000


In [12]:
pivot_df = long_df.pivot(index='Country', columns='Quarter', values='Sales')
print("\nPivot back to wide format:")
print(pivot_df)


Pivot back to wide format:
Quarter     Q1     Q2     Q3     Q4
Country                            
Canada    9000  11000  10000  13000
France   12000  14000  13000  16000
Germany  18000  16000  20000  25000
Spain    15000  17000  14000  20000
USA      21000  19000  22000  28000


In [13]:
pivot_df['Total'] = pivot_df.sum(axis=1)
pivot_df = pivot_df.sort_values('Total', ascending=False)
print("\nWith totals, sorted:")
print(pivot_df)


With totals, sorted:
Quarter     Q1     Q2     Q3     Q4  Total
Country                                   
USA      21000  19000  22000  28000  90000
Germany  18000  16000  20000  25000  79000
Spain    15000  17000  14000  20000  66000
France   12000  14000  13000  16000  55000
Canada    9000  11000  10000  13000  43000


In [14]:
df.to_csv("financials_clean.csv", index=False)
print("Saved!")
print(df.shape)

Saved!
(688, 26)
